In [0]:
# Importing the modules
#It defines the structure (schema) of your data
from pyspark.sql.types import * 
#Used for transformation. It provides the operations to actually calculate, filter, edit strings, and manipulate columns within your DataFrames.
from pyspark.sql.functions import *

In [0]:
#ADLS configuration (access to the storage account)
spark.conf.set(
  "fs.azure.account.key.Storageaccount_name.dfs.core.windows.net",
  <<"Storage_Access_Key">>
)



In [0]:
#specifying the part to write the data
bronze_path = "abfss://container@<<"Storageaccount_name">>.dfs.core.windows.net/path"
silver_path = "abfss://container@<<"Storageaccount_name">>.dfs.core.windows.net/path"


##patient_flow is a new container I just created that will be a folder inside bronze container

In [0]:
# Reading the stream that is in delta format from bronze
bronze_df = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
)

In [0]:
#Defining my Schema
schema = StructType([
    StructField("patient_id", StringType()),
    StructField("gender", StringType()),
    StructField("age", IntegerType()),
    StructField("department", StringType()),
    StructField("admission_time", StringType()),
    StructField("discharge_time", StringType()),
    StructField("bed_id", IntegerType()),
    StructField("hospital_id", IntegerType())
])

In [0]:
#Parse it to dataframe
parsed_df = bronze_df.withColumn("data",from_json(col("raw_json"),schema)).select("data.*")

In [0]:
#converting type to Timestamp
clean_df = parsed_df.withColumn("admission_time", to_timestamp("admission_time"))
clean_df = clean_df.withColumn("discharge_time", to_timestamp("discharge_time"))

In [0]:
#Invalid admission_times
clean_df = clean_df.withColumn("admission_time",
                               when(
                                   col("admission_time").isNull() | (col("admission_time") > current_timestamp()),
                                   current_timestamp())
                               .otherwise(col("admission_time")))

In [0]:
#Handling Invalid Age checking if age is more than 100 checking random age from 1 to 90 
clean_df = clean_df.withColumn("age",
                               when(col("age")>100,floor(rand()*90+1).cast("int"))
                               .otherwise(col("age"))
                               )

In [0]:
#schema evolution, assuming there is a collumn missing it creates a column with the missing column name and assighns the value as None, this is done for schema evolution handling
expected_cols = ["patient_id", "gender", "age", "department", "admission_time", "discharge_time", "bed_id", "hospital_id"]

for col_name in expected_cols:
    if col_name not in clean_df.columns:
        clean_df = clean_df.withColumn(col_name, lit(None))

In [0]:
# # Define clearly at the top
# silver_path       = "abfss://silver@hospitalstoragedon.dfs.core.windows.net/patient_flow"
# checkpoint_path   = "abfss://silver@hospitalstoragedon.dfs.core.windows.net/_checkpoints/patient_flow"

# # Then use them correctly
# (
#     clean_df.writeStream
#     .format("delta")
#     .outputMode("append")
#     .option("mergeSchema", "true")
#     .option("checkpointLocation", checkpoint_path)  # progress saved here
#     .start(silver_path)                              # data written here
# )

In [0]:
# # 1. Define distinct paths
# silver_table_path = "abfss://silver@hospitalstoragedon.dfs.core.windows.net/patient_flow"
# checkpoint_path = silver_table_path + "/_checkpoints"

# # 2. Run the stream
# (clean_df.writeStream
#     .format("delta")
#     .outputMode("append")
#     .option("mergeSchema", "true") 
#     .option("checkpointLocation", checkpoint_path) # Metadata goes here
#     .start(silver_table_path)                      # Data goes here
# )


In [0]:
#silver_path = "abfss://silver@hospitalstoragedon.dfs.core.windows.net/patient_flow"


In [0]:
#Writing to silver table
## .option("mergeSchema","true") But incase the schema changes it will not breake the pipeline but rather it will add the new column to the schema with the column that is added

#checkpoint_path = "abfss://silver@hospitalstoragedon.dfs.core.windows.net/_checkpoints/patient_flow"

(
    clean_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("mergeSchema","true")
    .option("checkpointLocation", silver_path + "_checkpoint")
    .start(silver_path)
)


In [0]:
## To see the records that is inserted to bronze table
display(spark.read.format("delta").load(silver_path))

patient_id,gender,age,department,admission_time,discharge_time,bed_id,hospital_id
497b056a-c0e1-49bd-bdeb-29532d50d3e9,Male,49,Cardiology,2026-04-19T14:00:37.941038Z,2026-04-22T06:00:37.941038Z,345,5
9013e599-aa05-40f0-b209-923e4f8ef2e9,Male,53,Maternity,2026-04-19T08:00:38.947885Z,2026-04-19T19:00:38.947885Z,291,5
c33365a1-49ff-4bee-bc3b-695c879ec891,Male,26,Cardiology,2026-04-17T21:00:39.949816Z,2026-04-18T00:00:39.949816Z,230,2
abe0110b-bb93-401f-b8a8-40dce360312e,Female,14,Surgery,2026-04-19T08:00:40.951727Z,2026-04-21T08:00:40.951727Z,61,1
bc53ada9-3511-4362-a37c-a4273edc434a,Male,95,Oncology,2026-04-18T08:00:41.953577Z,2026-04-20T23:00:41.953577Z,134,2
a9c5946a-c38c-42a6-8ee2-c83ce27b17de,Male,74,Emergency,2026-04-17T14:00:42.957535Z,2026-04-19T21:00:42.957535Z,193,2
0953e580-e47b-4083-b4f8-9b254c9fcf38,Female,17,ICU,2026-04-19T08:00:43.959919Z,2026-04-19T10:00:43.959919Z,475,3
3fd87634-204b-4306-9004-cbc4e2c4d353,Male,37,Maternity,2026-04-18T11:00:44.970309Z,2026-04-18T15:00:44.970309Z,303,1
8d774b05-17f0-4e61-8e67-31cdedf93d4e,Male,72,Surgery,2026-04-18T05:00:45.97537Z,2026-04-19T15:00:45.97537Z,308,4
87f41406-c7a3-4553-a4fc-15f86841e631,Female,12,Surgery,2026-04-17T19:00:46.981967Z,2026-04-19T16:00:46.981967Z,266,6
